In [ ]:
import pymupdf4llm
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
base_dir = Path().resolve().parent

In [6]:
# targeted pdf location
PDF_PATH = f'{base_dir}/data/raw/L5_DT.pdf'

In [7]:
# convert pdf to md
md_text = pymupdf4llm.to_markdown(PDF_PATH)

Path(f'{base_dir}/data/processed/L5_DT.md').write_bytes(md_text.encode())

13855

In [9]:
# divide text into chunks of 500 strings
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=25)
chunks = text_splitter.split_text(md_text)

In [10]:
print(len(chunks))
for c in chunks:
    len(c)

30


In [11]:
#defining the model 
model = SentenceTransformer('all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1578.72it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
embeddings = model.encode(chunks)

In [13]:
client = chromadb.PersistentClient(path=f'{base_dir}/data/vector')
collection = client.create_collection(name='my_collection')

In [15]:
collection.add(
    ids=[f"id{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
    metadatas=[{"Lecture": 3} for _ in range(len(chunks))]
)